# S2 Pilot runner — near-duplicate removal + the ARI trap-check

**This notebook is a RUNNER. It contains no logic.** All computation lives in
`src/cluster/s2_pilot.py` and all settings in `configs/s2_pilot.yaml`. If you find
yourself editing a cell to change behaviour, stop — change the config in the repo
and re-clone, or the result cannot enter the paper.

## Before you press Run

| Setting | Value | Why |
|---|---|---|
| Accelerator | **GPU (T4 or P100)** | LaBSE on 4,730 short reviews: seconds on GPU, ~10 min on CPU |
| Internet | **ON** | LaBSE weights download from HuggingFace |
| Input dataset | `bn_clean.csv` uploaded as a **private** Kaggle Dataset | it is gitignored, so the clone does not include it |

**Uploading the input:** Kaggle → *Datasets* → *New Dataset* → upload
`data/cleaned/bn_clean.csv` → keep it **Private** (it is derived from a licensed
corpus) → then *Add Input* on this notebook. Set `KAGGLE_INPUT_DIR` below to the
mounted path.

## Environment policy (decided 2026-07-30)

This run uses **Kaggle's host-native torch/CUDA** and installs only the packages
S2 actually needs. It does **not** apply `requirements.lock.txt`, which was frozen
on Windows and would drag in a full torch build that conflicts with Kaggle's CUDA
image.

The consequence is recorded, not hidden: the last cell writes
`results/env_snapshot_s2_kaggle.json` describing exactly what ran here, and
**that** file — not `requirements.lock.txt` — is the environment S2's numbers are
attributable to. The thesis must report both environments and say which produced
which result. `--out` guarantees the committed lock file is not overwritten from
this host.

## What this does NOT do

It writes **no split map**. `data/splits/split_map_v1.json` is created in a later
step, deliberately after the trap-check outcome is on record, so the split cannot
be tuned to it.

## 1. Clone the repo

In [ ]:
# If the repo is PRIVATE this clone fails. Do NOT paste a token into a cell:
# use Kaggle Secrets (Add-ons -> Secrets), or make the repo public for the run.
REPO_URL = "https://github.com/alphapie77/BSc_Thesis.git"

# EDIT THIS to the mounted path of the Kaggle Dataset holding bn_clean.csv.
KAGGLE_INPUT_DIR = "/kaggle/input/bn-clean"

!rm -rf /kaggle/working/thesis
!git clone --depth 1 $REPO_URL /kaggle/working/thesis
%cd /kaggle/working/thesis
!git log --oneline -1

## 2. Install only what S2 needs

`sentence-transformers` pulls `transformers` and uses the torch already present
on the host. Nothing here upgrades torch — that is the whole point.

In [ ]:
!pip install -q sentence-transformers pyyaml

# Confirm the GPU is actually visible. If this says False, fix the accelerator
# setting before continuing -- a CPU run is not wrong, just slow, but you should
# know which one you got because it is recorded in the snapshot.
import torch
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))

## 3. Bring in `bn_clean.csv` and verify it is the expected file

The row count is asserted **before** anything expensive runs. `bn_clean.csv` must
not be regenerated — `review_id`s are referenced by the split map — so a mismatch
here is a stop-and-investigate, not a number to update.

In [ ]:
import hashlib, shutil
from pathlib import Path

src = Path(KAGGLE_INPUT_DIR) / "bn_clean.csv"
assert src.exists(), (
    f"{src} not found. Add the Kaggle Dataset as an input and fix "
    f"KAGGLE_INPUT_DIR. Contents of /kaggle/input: "
    f"{[p.name for p in Path('/kaggle/input').glob('*')]}"
)

dst = Path("data/cleaned/bn_clean.csv")
dst.parent.mkdir(parents=True, exist_ok=True)
shutil.copy(src, dst)

# Record the input's identity so this run can be tied to a specific CSV later.
digest = hashlib.sha256(dst.read_bytes()).hexdigest()
print("sha256:", digest)
# The CSV produced on Windows on 2026-07-27. A DIFFERENT hash is not
# automatically wrong -- re-running s1_clean.py now writes LF instead of
# CRLF, which changes the hash while leaving every row identical. The row
# count assertion below is the check that actually matters.
EXPECTED = "929bd589e1cd1293ef5e3943ea0a7f5a7b1172016df21990acb53b8f807027c5"
print("matches the 2026-07-27 Windows build:", digest == EXPECTED)
print("bytes :", dst.stat().st_size)

# Cheap pre-flight: the same assertions the script makes, but before LaBSE
# downloads ~1.8 GB of weights. Failing here costs seconds, not a session.
import sys, yaml
sys.path.insert(0, ".")
from src.cluster.s2_pilot import load_clean

cfg = yaml.safe_load(Path("configs/s2_pilot.yaml").read_text(encoding="utf-8"))
df = load_clean(cfg, Path("."))
print("pre-flight OK:", len(df), "rows |",
      df[cfg["label_col"]].value_counts().sort_index().to_dict())

## 4. Run the tests before the experiment

Both suites are fast and neither needs a GPU. `test_s2_verdict.py` pins the
verdict bands to the pre-registration in `protocol.md`; `test_s2_numeric.py`
checks the blocked-matmul dedup against a brute-force reference. If either fails,
**do not read the report that follows** — the machinery producing it is broken.

In [ ]:
!python tests/test_s2_verdict.py && python tests/test_s2_numeric.py

## 5. Run S2

One config, one script, one result file.

In [ ]:
!python -m src.cluster.s2_pilot --config configs/s2_pilot.yaml

## 6. Record the environment that produced the run

`--out` is mandatory here. Without it this would overwrite the committed
`requirements.lock.txt` with a Linux freeze and destroy the record of the
environment every earlier step ran in. The script refuses `--out
requirements.lock.txt` outright.

In [ ]:
!python src/common/env_snapshot.py \
    --out results/env_snapshot_s2_kaggle.json \
    --note "Kaggle host-native torch/CUDA; S2 pilot run"

## 7. Read the report — **in this order**

The order matters. Reading the ARI first is how you talk yourself into a
conclusion the data does not support.

1. **Off-diagonal cosine distribution.** Median review length is 8 words, so
   unrelated reviews already sit at high cosine. If the 99.9th percentile is
   *above* a swept threshold, that threshold is cutting into the bulk of the
   distribution — it is removing merely *similar* short reviews, not duplicates,
   and the choice needs defending before anything else is read.
2. **Degeneracy.** Any cluster below 5% or above 70% of *n* means K-Means did not
   partition. Then the verdict is `NO_CLAIM` (Band 0) and **the ARI is
   uninterpretable in either direction** — a non-partition scores low ARI by
   construction and must never be read as "personas are independent of
   sentiment".
3. **ARI and its band**, then χ² and Cramér's V *together with* it. At n ≈ 4,700
   χ²'s p-value reaches significance on associations far too weak to matter. High
   Cramér's V with low ARI means the clusters lean on sentiment without
   reproducing its partition — a caveat, not a pass.
4. **The sensitivity curve.** If the `Verdict` column is *not* constant across
   0.90 / 0.95 / 0.98, the conclusion depends on an arbitrary threshold and must
   be reported that way.

Whatever comes out is the result. All four bands have a pre-committed claim in
`docs/protocol.md` (RQ1, registered 2026-07-28, before any ARI existed), so a
bad number is still publishable. Re-running with different settings to move the
number destroys the only honest test in the thesis.

In [ ]:
from IPython.display import Markdown, display
from pathlib import Path

display(Markdown(Path("results/s2_pilot_ari_trapcheck.md").read_text(encoding="utf-8")))

## 8. Package the outputs for download

Download `s2_outputs.zip` from the Kaggle output pane, unzip into the repo on
your laptop, and commit **with** the lab-notebook entry — `.githooks/pre-commit`
blocks a `results/` change that arrives without its reasoning.

`near_dup_pairs.csv` is gitignored (derived data) but keep it: it is how a
removal gets audited by eye, and it carries both review texts for every pair.

In [ ]:
import shutil, zipfile
from pathlib import Path

OUT = Path("/kaggle/working/s2_outputs.zip")
wanted = [
    "results/s2_pilot_ari_trapcheck.md",
    "results/env_snapshot_s2_kaggle.json",
    "data/cleaned/near_dup_pairs.csv",
]
with zipfile.ZipFile(OUT, "w", zipfile.ZIP_DEFLATED) as z:
    for f in wanted:
        p = Path(f)
        if p.exists():
            z.write(p, f)
            print(f"added {f} ({p.stat().st_size:,} bytes)")
        else:
            print(f"MISSING {f}")
print(f"\nwrote {OUT}")

# The LaBSE embedding cache is large and fully reproducible from the CSV, so it
# is deliberately NOT in the zip. Re-embedding is cheap; storing it is not.
emb = Path("data/cleaned/labse_emb_bn_clean.npy")
if emb.exists():
    print(f"note: {emb} is {emb.stat().st_size / 1e6:.0f} MB and was not zipped "
          "(reproducible from bn_clean.csv).")